# UNO: known-causal-effect simulation benchmark

This notebook runs the source-preserving scripts in `simulation_benchmark/scripts/`.
It uses fully simulated data and does not require clinical data or the existing
repository demo files. **Restart the kernel, then run the cells in order.**

The source's numerical parameters, random seeds, random-number call order,
optimizer settings, selection rules, and estimators are retained. The original
notebook is archived under `reference/`; it is not used to supply fitted values.

**Default scope:** primary benchmark plus the source's current **100%** development-NCO
replacement setting. A separate, explicitly enabled cell replays all six fractions
shown in the source's accumulated output: 10%, 25%, 50%, 75%, 90%, and 100%.

**Reproducibility:** use `environment.yml` / `requirements.txt` to match the recorded
software versions as closely as possible. Matching seeds alone is not a verified
promise of matching archived outputs in a different software or hardware environment.
The final cells compare fresh results with saved displays and export the actual run.

In [1]:
from pathlib import Path
import sys

# Support launching Jupyter from this folder or from the repository root.
_uno_candidates = (Path.cwd(), Path.cwd() / "simulation_benchmark")
_uno_root = next((p.resolve() for p in _uno_candidates
                  if (p / "workflow.py").is_file()
                  and (p / "reference/source_manifest.json").is_file()), None)
if _uno_root is None:
    raise FileNotFoundError("Open Jupyter in simulation_benchmark or its parent repository folder.")
if str(_uno_root) not in sys.path:
    sys.path.insert(0, str(_uno_root))

from workflow import run_stage, SENSITIVITY_STAGES
from verify_source import verify_source

# Verifies all scientific statements and their original order, before fitting.
_uno_source_check = verify_source()
print("Scientific source preserved:", _uno_source_check["scientific_ast_equivalent"])
print("Nonempty original code cells:", _uno_source_check["nonempty_original_code_cells"])
print("Benchmark directory:", _uno_root)

Scientific source preserved: True
Nonempty original code cells: 31
Benchmark directory: /Users/yuewudb/UNO_revision_ex/benchmark/simulation_benchmark


## 1. Imports and unchanged numerical helpers

Load the network classes, nonlinear score functions, IPW risk-ratio estimator,
Gaussian-null fitting, and EASE. Retain one CPU thread, deterministic Torch
algorithms, the original dtypes, and the post-ReLU activation score.

In [2]:
run_stage("01_setup", globals())

{'python': '3.11.0', 'numpy': '1.26.4', 'pandas': '2.2.3', 'scipy': '1.11.4', 'torch': '2.5.1', 'sklearn': '1.2.2', 'statsmodels': '0.14.0'}


## 2. Original data-generating mechanism

Generate 10,000 individuals, 20 observed covariates, five unmeasured variables,
and 80 NCOs; retain the original 40/40 development/test split.

The executable NCO-parameter seed is **2026**, not the value mentioned in an
outdated source markdown cell. The initial outcome uses `Y_GAMMA_U = 0.2`;
the final target-outcome override is intentionally retained in section 8.
This initial outcome is not used for propensity-score training or NCO selection.

In [3]:
run_stage("02_generate_data", globals())

X64, U shapes: (10000, 20) (10000, 5)


,score,sample_mean,sample_sd
0,C(X),-0.226824,0.974241
1,S(X),-0.252741,1.079201
2,V(U),-0.075072,0.793616


A prevalence: 0.4028 | true gate proportion: 0.3007
e_true_xu quantiles: [0.05011401 0.05475955 0.354548   0.93793454 0.94990217]
W shape: (10000, 80) | prevalence range: 0.065 0.1999
True marginal RR: 0.75
Mean risks under A=0 / A=1: 0.0807447583833364 0.06055856878750231
Observed Y events: 716
Xt, At: torch.Size([10000, 20]) torch.Size([10000, 1])
Eligible / development / held-out NCOs: 80 40 40


## 3. Logistic-regression propensity scores

Fit the original observed-covariate logistic comparator. Do not supply U.

In [4]:
run_stage("03_logistic_ps", globals())

Logistic treatment accuracy: 0.6414
Logistic PS range: 0.07180967155903815 0.8294408205808129


## 4. Base neural network

Initialize and fit the original 100-by-100-by-100 ReLU network for up to 50 epochs.
Retain the original initialization, batch order, dropout, Adam settings, learning-rate
schedule, accuracy-based stopping rule, fitted weights, and post-training RNG state.
The training loop remains explicit in the script.

In [5]:
run_stage("04_base_model", globals())

PS(
  (stack): ModuleList(
    (0): Linear(in_features=20, out_features=100, bias=True)
    (1-2): 2 x Linear(in_features=100, out_features=100, bias=True)
    (3): Linear(in_features=100, out_features=1, bias=True)
  )
  (act): ReLU()
  (dropout): Dropout(p=0.1, inplace=False)
  (sigmoid): Sigmoid()
)
Base 01/50  loss=0.64818  accuracy=0.6592  lr=0.001
Base 02/50  loss=0.62475  accuracy=0.6746  lr=0.001
Base 03/50  loss=0.61690  accuracy=0.6836  lr=0.001
Base 04/50  loss=0.60758  accuracy=0.6877  lr=0.001
Base 05/50  loss=0.60298  accuracy=0.6998  lr=0.001
Base 06/50  loss=0.59445  accuracy=0.7068  lr=0.001
Base 07/50  loss=0.59106  accuracy=0.7135  lr=0.001
Base 08/50  loss=0.58453  accuracy=0.7143  lr=0.001
Base 09/50  loss=0.57788  accuracy=0.7211  lr=0.001
Base 10/50  loss=0.57214  accuracy=0.7338  lr=0.0005
Base 11/50  loss=0.55793  accuracy=0.7442  lr=0.0005
Base 12/50  loss=0.55009  accuracy=0.7493  lr=0.0005
Base 13/50  loss=0.54775  accuracy=0.7538  lr=0.0005
Base 14/50  loss

## 5. Development-NCO signals and groups

Fit the original treated/control logistic models. Preserve both the threshold
scan and the subsequent explicit **`threshold_used = 0.08`** override.
Group membership follows the original count cutoff; it is not forced to equal 15%.

In [6]:
run_stage("05_nco_signals", globals())

Development NCO models: 10/40
Development NCO models: 20/40
Development NCO models: 30/40
Development NCO models: 40/40
Automatic signal-threshold calibration (development signals only):


,signal_threshold,count_cutoff,high_fraction,distance_to_target
0,0.079516,16,0.1505,0.0005
1,0.079881,16,0.1492,0.0008
2,0.079152,16,0.1522,0.0022
3,0.080246,16,0.1470,0.0030
4,0.078787,16,0.1540,0.0040


NCO signal threshold used: 0.08 | count cutoff: 16.0
High / low counts: 1484 8516
Actual high proportion: 14.84%
Shapes: torch.Size([1484, 20]) torch.Size([8516, 20]) torch.Size([1484, 1]) torch.Size([8516, 1])


## 6. Original UNO pruning and fine-tuning

Retain candidate fractions **[0.35, 0.40, 0.45, 0.50]** and 15 fine-tuning epochs.
Every candidate starts from the fitted Base weights. Preserve global top-k scoring,
incoming/bias/outgoing masks, gradient masking, the low-signal training subset,
and selection using development NCOs only. The original Base is unchanged.

In [7]:
run_stage("06_uno", globals())

UNO starts from fitted Base: {'in_N': 20, 'm': 100, 'depth': 3, 'dropout': 0.1, 'ps_floor': 0.01}
ratio=0.35: removed=[33, 31, 41], total=105
  UNO 01/15  loss_low=0.59539  accuracy_all=0.7228
  UNO 02/15  loss_low=0.57159  accuracy_all=0.7315
  UNO 03/15  loss_low=0.56768  accuracy_all=0.7269
  UNO 04/15  loss_low=0.56299  accuracy_all=0.7395
  UNO 05/15  loss_low=0.55362  accuracy_all=0.7466
  UNO 06/15  loss_low=0.55083  accuracy_all=0.7479
  UNO 07/15  loss_low=0.55256  accuracy_all=0.7484
  UNO 08/15  loss_low=0.54474  accuracy_all=0.7568
  UNO 09/15  loss_low=0.54352  accuracy_all=0.7603
  UNO 10/15  loss_low=0.53817  accuracy_all=0.7606
  UNO 11/15  loss_low=0.52302  accuracy_all=0.7734
  UNO 12/15  loss_low=0.51748  accuracy_all=0.7737
  UNO 13/15  loss_low=0.51833  accuracy_all=0.7769
  UNO 14/15  loss_low=0.50720  accuracy_all=0.7754
  UNO 15/15  loss_low=0.50659  accuracy_all=0.7801
  development EASE: 0.21502161100133618
ratio=0.40: removed=[37, 36, 47], total=120
  UNO 01/

,ratio,pruning_only_dev_EASE,UNO_dev_EASE
0,0.35,0.440166,0.215022
1,0.40,0.495105,0.192191
2,0.45,0.502365,0.179602
3,0.50,0.471195,0.233214


,ratio,start_widths,pruned,retained
0,0.35,"[100, 100, 100]","[33, 31, 41]","[67, 69, 59]"
1,0.40,"[100, 100, 100]","[37, 36, 47]","[63, 64, 53]"
2,0.45,"[100, 100, 100]","[42, 40, 53]","[58, 60, 47]"
3,0.50,"[100, 100, 100]","[48, 44, 58]","[52, 56, 42]"


## 7. Held-out NCO evaluation

Evaluate the same 40 held-out NCOs for all original comparators, including the true-PS oracle.

In [8]:
run_stage("07_heldout_evaluation", globals())

,method,n_heldout_ncos,heldout_EASE,null_mean,null_sd
0,Unadjusted,40,0.535613,0.535613,0.037922
1,Logistic PS,40,0.348938,0.348938,0.051016
2,Base NN,40,0.250688,0.250688,0.044391
3,Pruning only,40,0.495193,0.495193,0.065958
4,UNO,40,0.190230,0.190213,0.058329
5,"Oracle true PS (X,U)",40,0.071144,-0.063987,0.056233


## 8. Final target outcome and single-realization RR

Preserve the original later override: **`Y_GAMMA_U = 2`**, **`Y_USE_V1 = True`**.
The quadratic score uses all five unmeasured variables and cohort-fixed normalization.
The true conditional and marginal causal RR is **0.75**.

The source retains the initial outcome in `df["Y"]`; downstream target-effect
calculations explicitly use the later `Y` variable. This distinction is preserved,
not silently repaired or reordered. There is no trimming or extra post-fit clipping.

In [9]:
run_stage("08_target_outcome", globals())

True marginal RR: 0.75
Mean risks under A=0 / A=1: 0.12130943934383093 0.0909820795078732
Observed Y events: 1053


,method,n_used,RR_hat,true_RR,RR_error,abs_RR_error,log_rr_se_model
0,Unadjusted,10000,1.106434,0.75,0.356434,0.356434,0.062294
1,Logistic PS,10000,0.944480,0.75,0.194480,0.194480,0.043579
2,Base NN,10000,0.854634,0.75,0.104634,0.104634,0.048270
3,Pruning only,10000,1.030579,0.75,0.280579,0.280579,0.040251
4,UNO,10000,0.709005,0.75,-0.040995,0.040995,0.045365
5,"Oracle true PS (X,U)",10000,0.683581,0.75,-0.066419,0.066419,0.044005


## 9. NCO-only empirical calibration

Fit the Gaussian systematic-error model using development NCOs for Base NN and
Logistic PS. The point-estimate adjustment subtracts the fitted null mean on the
log-RR scale. It does not modify propensity scores or weights. EC rows retain the
parent model's pre-calibration held-out EASE.

In [10]:
run_stage("09_empirical_calibration", globals())

Same current Y: single-run RR and estimation errors (not repeated-Y averages):


,method,heldout_EASE_before_EC,n_used,true_RR,RR_hat,RR_error,abs_RR_error
0,Unadjusted,0.535613,10000,0.75,1.106434,0.356434,0.356434
1,Logistic PS,0.348938,10000,0.75,0.944480,0.194480,0.194480
2,Base NN,0.250688,10000,0.75,0.854634,0.104634,0.104634
3,Pruning only,0.495193,10000,0.75,1.030579,0.280579,0.280579
4,UNO,0.190230,10000,0.75,0.709005,-0.040995,0.040995
5,"Oracle true PS (X,U)",0.071144,10000,0.75,0.683581,-0.066419,0.066419
6,Base NN + EC (NCO-only),0.250688,10000,0.75,0.668413,-0.081587,0.081587
7,Logistic PS + EC (NCO-only),0.348938,10000,0.75,0.671795,-0.078205,0.078205


Same current Y: calibrated intervals (95%):


,method,RR_before_EC,RR_EC,CI_lower_EC,CI_upper_EC,p_EC
0,Base NN + EC (NCO-only),0.854634,0.668413,0.587412,0.760585,9.831646e-10
1,Logistic PS + EC (NCO-only),0.944480,0.671795,0.593868,0.759948,2.555095e-10


Development-NCO calibration parameters (log-RR scale):


,method,n_development_NCOs,mu_full,tau_full,mu_RR,tau_RR,n_RR_patients
0,Base NN + EC (NCO-only),40,0.245766,0.044879,0.245766,0.044879,10000
1,Logistic PS + EC (NCO-only),40,0.340681,0.045368,0.340681,0.045368,10000


Single-run errors are not Bias/MAE/RMSE across repeated datasets.
CIs retain the original model-based SE and treat calibration parameters as fixed.


## 10. Paired target-outcome realizations

Generate the original **1,000 outcome-only realizations**, holding the cohort and
fitted models fixed. Use the same realization for every method. Retain all original
RR, MAE, RMSE, conditional-bias, calibration, and paired-comparison calculations.
These are not 1,000 independently generated cohorts or model refits.

In [11]:
run_stage("10_repeated_outcomes", globals())

,method,heldout_EASE,mean_rr,bias_rr,mae_rr,rmse_rr
0,Unadjusted,0.535613,1.100114,0.350114,0.350114,0.353798
1,Logistic PS,0.348938,0.949849,0.199849,0.199849,0.205205
2,Base NN,0.250688,0.875081,0.125081,0.125311,0.135511
3,Pruning only,0.495193,1.093173,0.343173,0.343173,0.348571
4,UNO,0.190230,0.803900,0.053900,0.059480,0.071769
5,"Oracle true PS (X,U)",0.071144,0.726909,-0.023091,0.039588,0.048693


Maximum weighted-Poisson check error: 7.771561172376096e-15
Outcome-only repeats; independent full datasets = 1.
Same Y replicates for all methods; EC uses fixed development-NCO parameters:


,method,n_y_replicates,heldout_EASE_before_EC,mean_rr,bias_rr,mae_rr,rmse_rr
0,Unadjusted,1000,0.535613,1.100114,0.350114,0.350114,0.353798
1,Logistic PS,1000,0.348938,0.949849,0.199849,0.199849,0.205205
2,Base NN,1000,0.250688,0.875081,0.125081,0.125311,0.135511
3,Pruning only,1000,0.495193,1.093173,0.343173,0.343173,0.348571
4,UNO,1000,0.190230,0.803900,0.053900,0.059480,0.071769
5,"Oracle true PS (X,U)",1000,0.071144,0.726909,-0.023091,0.039588,0.048693
6,Base NN + EC (NCO-only),1000,0.250688,0.684405,-0.065595,0.067403,0.077235
7,Logistic PS + EC (NCO-only),1000,0.348938,0.675614,-0.074386,0.074778,0.081430


The first EC replicate matches the single-Y EC result.
EC rows inherit the original held-out EASE of their parent methods.
Outcome-only repeats; independent full datasets = 1.


,comparison,fraction_UNO_lower_abs_error,MAE_difference_UNO_minus_baseline,MSE_difference_UNO_minus_baseline,relative_RMSE_reduction
0,UNO vs Logistic PS,0.998,-0.140369,-0.036958,0.650254
1,UNO vs Base NN,0.965,-0.065831,-0.013212,0.470378


## 11. Original sensitivity parameters

The uploaded source currently sets **`NCO25_REPLACEMENT_FRACTION = 1`** (100%).
The inherited `NCO25_` names are retained for source compatibility; they do not
force the replacement fraction to be 25%. Both perturbation seeds remain 2026.

In [13]:
run_stage("11_sensitivity_parameters", globals())

## 12. Development-NCO perturbation

Regenerate only the selected development NCOs with their direct U coefficients
set to zero. Retain all other coefficients without prevalence matching. The
remaining development NCOs and all held-out NCOs remain unchanged. No new
cohort, Base model, or target-outcome realizations are created.

For a deliberate alternative setting, change `NCO25_REPLACEMENT_FRACTION`
**after section 11**, then run sections 12–14. To preserve the source default,
leave it at 1. All replacement subsets are drawn using the original code.

In [14]:
run_stage("12_sensitivity_data", globals())

Fixed cohort: n=10000; existing Y realizations=1000
Replace 40/40 development NCOs; retain all 40 held-out NCOs.
Original Base parameters, predictions, data and Y realizations will not be overwritten.


,nco,df_column_index,replaced,original_U_loading,scenario_U_loading,prevalence_original,prevalence_scenario,prevalence_treated,prevalence_control,estimable
0,W38,58,True,0.988957,0.0,0.1122,0.0967,0.128352,0.075352,True
1,W65,85,True,1.110496,0.0,0.1815,0.1702,0.217478,0.138312,True
2,W37,57,True,1.352575,0.0,0.1297,0.1047,0.130338,0.087408,True
3,W04,24,True,1.290683,0.0,0.1074,0.0868,0.106504,0.073510,True
4,W54,74,True,1.108142,0.0,0.1191,0.1007,0.131082,0.080208,True
5,W01,21,True,1.177141,0.0,0.0802,0.0639,0.073982,0.057100,True
6,W41,61,True,1.287191,0.0,0.1512,0.1290,0.167080,0.103315,True
7,W79,99,True,1.082868,0.0,0.0650,0.0547,0.075472,0.040690,True
8,W56,76,True,1.040880,0.0,0.1366,0.1199,0.154419,0.096618,True
9,W32,52,True,1.174736,0.0,0.1801,0.1568,0.201092,0.126926,True


Unchanged: all non-replaced development NCOs and all held-out NCO observations.


## 13. Refit the NCO-guided modification

Repeat NCO prediction, grouping, scoring, pruning, and fine-tuning using the
original rules and candidate fractions. The actual 0.08 signal threshold is
inherited; the count cutoff is recomputed. A degenerate split is reported as an
error rather than silently changing the rule. Development EASE selects the model.

In [15]:
run_stage("13_sensitivity_development", globals())

NCO25: fitted 10/40 development NCO model pairs
NCO25: fitted 20/40 development NCO model pairs
NCO25: fitted 30/40 development NCO model pairs
NCO25: fitted 40/40 development NCO model pairs
Estimated signals may be nonzero even for NCOs with no direct U dependence.


,replaced,n_ncos,mean_absolute_estimated_signal
0,True,40,0.034198


,signal_threshold,count_offset,count_cutoff,n_high,n_low,high_fraction
0,0.08,3,13,566,9434,0.0566


NOTE: high fraction is outside the original reference range; rule remains unchanged.
Inherited Base architecture: {'in_N': 20, 'm': 100, 'depth': 3, 'dropout': 0.1, 'ps_floor': 0.01}
Candidate ratios: (0.35, 0.4, 0.45, 0.5) | fine-tuning epochs: 15
NCO25 ratio=0.35; removed per layer=[36, 30, 39]
  01/15 loss_low=0.61144 accuracy_all=0.7016
  02/15 loss_low=0.58701 accuracy_all=0.7278
  03/15 loss_low=0.57562 accuracy_all=0.7345
  04/15 loss_low=0.57461 accuracy_all=0.7367
  05/15 loss_low=0.56364 accuracy_all=0.7494
  06/15 loss_low=0.55713 accuracy_all=0.7409
  07/15 loss_low=0.55560 accuracy_all=0.7580
  08/15 loss_low=0.55155 accuracy_all=0.7598
  09/15 loss_low=0.54775 accuracy_all=0.7643
  10/15 loss_low=0.54611 accuracy_all=0.7655
  11/15 loss_low=0.53372 accuracy_all=0.7727
  12/15 loss_low=0.52720 accuracy_all=0.7810
  13/15 loss_low=0.52570 accuracy_all=0.7755
  14/15 loss_low=0.52304 accuracy_all=0.7850
  15/15 loss_low=0.52085 accuracy_all=0.7873
  development EASE: 0.16947

,ratio,development_EASE,removed_per_layer,retained_per_layer
0,0.35,0.169475,"[36, 30, 39]","[64, 70, 61]"
1,0.40,0.136115,"[45, 34, 41]","[55, 66, 59]"
2,0.45,0.131261,"[49, 42, 44]","[51, 58, 56]"
3,0.50,0.187669,"[55, 46, 49]","[45, 54, 51]"


## 14. Evaluate and collect the current sensitivity setting

Compare only the fixed Base Model and this setting's UNO using the original
held-out NCOs and paired outcomes. Integrity checks confirm that the original
arrays, data frame, Base weights, and original UNO predictions are unchanged.

In [16]:
run_stage("14_sensitivity_evaluation", globals())

True RR=0.75; same 1000 paired outcome realizations


,method,heldout_EASE,mean_rr,mae_rr,rmse_rr
0,Base Model,0.250688,0.875081,0.125311,0.135511
1,UNO (100% development NCOs without U),0.236322,0.870925,0.120996,0.129597


,comparison,MAE_difference,MSE_difference,relative_RMSE_reduction,fraction_UNO_lower_absolute_error
0,NCO25 UNO minus Base Model,-0.004315,-0.001568,0.043636,0.56


Original data, fitted Base, original UNO predictions (if present), and Y realizations: unchanged.
One fixed cohort and one specified NCO perturbation; only the original Y realizations are repeated.


,no_U_percent,method,heldout_EASE,mean_rr,mae_rr,rmse_rr
0,100,Base Model,0.250688,0.875081,0.125311,0.135511
1,100,UNO (100% development NCOs without U),0.236322,0.870925,0.120996,0.129597


## 15. Optional: replay all six recorded replacement fractions

The original notebook accumulated these rows by manually rerunning its sensitivity
cells. A fresh run of its saved code only evaluates 100%. Enable the cell below to
replay **10%, 25%, 50%, 75%, 90%, and 100%** using the same Base and the original
sensitivity scripts. No other scientific parameters are changed. Replacement sets
are selected separately, not forced to be nested.

Each completed scenario can be saved with its actual weights and predictions.
This output option changes file saving only, not the analysis. The default is
**False** for the replay itself, preserving the saved source's single-setting run.

In [18]:
RUN_FULL_NCO_SENSITIVITY = True #False
SAVE_EACH_SENSITIVITY_RUN = True

if RUN_FULL_NCO_SENSITIVITY:
    from benchmark_io import save_run
    for _uno_fraction in (0.10, 0.25, 0.50, 0.75, 0.90, 1.0):
        NCO25_REPLACEMENT_FRACTION = _uno_fraction
        print(f"\n--- Replacement fraction: {100 * _uno_fraction:g}% ---")
        # Do not rerun stage 11, which deliberately preserves the source default.
        for _uno_stage in SENSITIVITY_STAGES:
            run_stage(_uno_stage, globals())
        if SAVE_EACH_SENSITIVITY_RUN:
            _uno_saved_fraction = save_run(globals())
            print("Saved this completed setting:", _uno_saved_fraction)
else:
    print("Full fraction replay disabled; the source-default 100% result is retained.")


--- Replacement fraction: 10% ---
Fixed cohort: n=10000; existing Y realizations=1000
Replace 4/40 development NCOs; retain all 40 held-out NCOs.
Original Base parameters, predictions, data and Y realizations will not be overwritten.


,nco,df_column_index,replaced,original_U_loading,scenario_U_loading,prevalence_original,prevalence_scenario,prevalence_treated,prevalence_control,estimable
0,W79,99,True,1.082868,0.0,0.0650,0.0560,0.072244,0.045044,True
1,W56,76,True,1.040880,0.0,0.1366,0.1247,0.163853,0.098292,True
2,W39,59,True,1.044396,0.0,0.1651,0.1728,0.208044,0.149029,True
3,W57,77,True,1.035340,0.0,0.1282,0.1136,0.151440,0.088078,True


Unchanged: all non-replaced development NCOs and all held-out NCO observations.
NCO25: fitted 10/40 development NCO model pairs
NCO25: fitted 20/40 development NCO model pairs
NCO25: fitted 30/40 development NCO model pairs
NCO25: fitted 40/40 development NCO model pairs
Estimated signals may be nonzero even for NCOs with no direct U dependence.


,replaced,n_ncos,mean_absolute_estimated_signal
0,False,36,0.045463
1,True,4,0.037264


,signal_threshold,count_offset,count_cutoff,n_high,n_low,high_fraction
0,0.08,3,16,1326,8674,0.1326


...

,method,heldout_EASE,mean_rr,mae_rr,rmse_rr
0,Base Model,0.250688,0.875081,0.125311,0.135511
1,UNO (100% development NCOs without U),0.236322,0.870925,0.120996,0.129597


,comparison,MAE_difference,MSE_difference,relative_RMSE_reduction,fraction_UNO_lower_absolute_error
0,NCO25 UNO minus Base Model,-0.004315,-0.001568,0.043636,0.56


Original data, fitted Base, original UNO predictions (if present), and Y realizations: unchanged.
One fixed cohort and one specified NCO perturbation; only the original Y realizations are repeated.


,no_U_percent,method,heldout_EASE,mean_rr,mae_rr,rmse_rr
0,10,Base Model,0.250688,0.875081,0.125311,0.135511
1,10,UNO (10% development NCOs without U),0.221743,0.830804,0.082201,0.093935
2,25,Base Model,0.250688,0.875081,0.125311,0.135511
3,25,UNO (25% development NCOs without U),0.219138,0.841341,0.092558,0.105061
4,50,Base Model,0.250688,0.875081,0.125311,0.135511
5,50,UNO (50% development NCOs without U),0.170971,0.805899,0.060585,0.072816
6,75,Base Model,0.250688,0.875081,0.125311,0.135511
7,75,UNO (75% development NCOs without U),0.235264,0.838985,0.090149,0.103054
8,90,Base Model,0.250688,0.875081,0.125311,0.135511
9,90,UNO (90% development NCOs without U),0.243006,0.823824,0.076417,0.090151


Saved this completed setting: /Users/yuewudb/UNO_revision_ex/benchmark/simulation_benchmark/results/run_20260926T180536515788Z_23447


## 16. Five-column tables and comparison with archived displays

The following tables are calculated from this run, not loaded as expected answers.
The original raw outputs (including additional comparators and metrics) remain
available in the notebook namespace and are exported separately. Archived display
values have only six decimal places and are used solely for this post-run audit.
A discrepancy is reported, never hidden or corrected by replacing computed values.

In [19]:
from benchmark_io import publication_tables, compare_saved_tables

benchmark_table1, benchmark_table2 = publication_tables(globals())
print("Table 1: primary benchmark (true RR = 0.75)")
display(benchmark_table1.round(4))
print("Table 2: development-NCO sensitivity settings completed in this kernel")
display(benchmark_table2.round(4))

benchmark_reference_check = compare_saved_tables(globals())
print("Fresh values match archived displays:", bool(benchmark_reference_check["matches_saved_display"].all()))
display(benchmark_reference_check.groupby("table", as_index=False).agg(
    matched_entries=("matches_saved_display", "sum"),
    total_entries=("matches_saved_display", "size"),
    largest_absolute_difference=("absolute_difference", "max"),
))

Table 1: primary benchmark (true RR = 0.75)


,method,heldout_EASE,mean_rr,mae_rr,rmse_rr
0,Unadjusted,0.5356,1.1001,0.3501,0.3538
1,Logistic PS,0.3489,0.9498,0.1998,0.2052
2,Base Model,0.2507,0.8751,0.1253,0.1355
3,UNO,0.1902,0.8039,0.0595,0.0718
4,Logistic PS + EC,0.3489,0.6756,0.0748,0.0814
5,Base Model + EC,0.2507,0.6844,0.0674,0.0772
6,"Oracle: true PS (X, U)",0.0711,0.7269,0.0396,0.0487


Table 2: development-NCO sensitivity settings completed in this kernel


,method,heldout_EASE,mean_rr,mae_rr,rmse_rr
0,Base Model,0.2507,0.8751,0.1253,0.1355
1,UNO (0%; original),0.1902,0.8039,0.0595,0.0718
2,UNO (10%),0.2217,0.8308,0.0822,0.0939
3,UNO (25%),0.2191,0.8413,0.0926,0.1051
4,UNO (50%),0.1710,0.8059,0.0606,0.0728
5,UNO (75%),0.2353,0.8390,0.0901,0.1031
6,UNO (90%),0.2430,0.8238,0.0764,0.0902
7,UNO (100%),0.2363,0.8709,0.1210,0.1296


Fresh values match archived displays: True


,table,matched_entries,total_entries,largest_absolute_difference
0,primary,32,32,4.991299e-07
1,sensitivity,48,48,4.921027e-07


## 17. Export the actual run

Save fresh tables, replicate-level results, simulated data, propensity scores,
model weights, masks, RNG states, actual parameters, and installed-package versions
to a new timestamped directory. No previous run is overwritten.

The same export call can be pasted into the **original running notebook kernel**
to preserve the fitted weights underlying its displayed results before restarting.
Saved notebook output alone is not a checkpoint.

In [ ]:
from benchmark_io import save_run
benchmark_output_dir = save_run(globals())
print("Archived this actual run:", benchmark_output_dir)